# Berlin EmoDB -> spectrogrammes RGB

Telecharge la base Berlin EmoDB et convertit chaque fichier wav en une image RGB de son mel-spectrogramme.

Pretraitement : 16 kHz mono, fenetre de 3 s, trames de 25 ms toutes les 10 ms,
64 bandes mel en dB, standardisees (z-score par extrait).

In [ ]:
# deps (deja installees dans .venv, selectionner le kernel "Python (CNN_RO11 .venv)")
%pip install -q librosa matplotlib numpy soundfile pillow nbstripout

# filtre git : retire les sorties et la metadata des .ipynb commites.
# A relancer une fois par clone (le filtre vit dans .git/config, qui n'est pas suivi).
import sys
!{sys.executable} -m nbstripout --install --attributes .gitattributes

In [ ]:
import io, zipfile, urllib.request
from pathlib import Path

import librosa
import numpy as np
from matplotlib import cm
from PIL import Image

URL = "http://emodb.bilderbar.info/download/download.zip"
RAW = Path("data/emodb")
OUT = Path("data/spectrograms")

# code emotion = 6eme caractere du nom de fichier (ex: 03a01Fa.wav -> F -> joie)
EMOTIONS = {
    "W": "anger", "L": "boredom", "E": "disgust", "A": "fear",
    "F": "happiness", "T": "sadness", "N": "neutral",
}

In [ ]:
# 1. telechargement + extraction
if not RAW.exists():
    RAW.mkdir(parents=True)
    print("downloading...")
    data = urllib.request.urlopen(URL).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall(RAW)

wavs = sorted(RAW.rglob("*.wav"))
print(len(wavs), "fichiers wav")

In [ ]:
# 2. wav -> mel-spectrogramme -> image RGB
SR = 16000              # 16 kHz mono
DURATION = 3.0          # fenetre de 3 s (padding / troncature)
N_FFT = int(0.025 * SR)  # trames de 25 ms  -> 400 echantillons
HOP = int(0.010 * SR)    # pas de 10 ms     -> 160 echantillons
N_MELS = 64             # 64 bandes mel
N_FRAMES = int(DURATION * SR / HOP)  # 300 trames

def wav_to_rgb(path):
    y, sr = librosa.load(path, sr=SR, mono=True)
    y = librosa.util.fix_length(y, size=int(DURATION * SR))  # 3 s exactement
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel)[:, :N_FRAMES]  # (64, 300)
    z = (db - db.mean()) / (db.std() + 1e-9)     # standardisation par extrait
    norm = np.clip(z, -3, 3) / 6 + 0.5           # +/-3 sigma -> 0..1
    rgb = (cm.viridis(norm[::-1])[..., :3] * 255).astype(np.uint8)  # basses freq en bas
    return Image.fromarray(rgb)

for wav in wavs:
    label = EMOTIONS[wav.stem[5]]
    dest = OUT / label
    dest.mkdir(parents=True, exist_ok=True)
    wav_to_rgb(wav).save(dest / f"{wav.stem}.png")

print("images ecrites dans", OUT)

In [ ]:
# 3. verification rapide
for d in sorted(OUT.iterdir()):
    print(d.name, len(list(d.glob("*.png"))))

display(Image.open(next(OUT.rglob("*.png"))))